# Exercise 2, Step 1 — Prepare the derived context

Loads `data/winequality-red.csv`, draws a 200-row sample stratified by quality
tier, and exports:

- `data/wine_sample_200.csv` — raw sampled rows (traceability)
- `data/wine_toscana.sql` — the **raw many-valued** context as HSQLDB
  `INSERT`s. Elba builds the conceptual scales itself (Ordinal/Interordinal
  scale dialogs with thresholds), so this file intentionally carries the
  original numeric values, not pre-binarized columns.

The scale functions below are still used to build a Python-side derived
binary context for `ex2_implications.ipynb` and to verify scale closure;
their thresholds are exactly what was typed into Elba's scale dialogs.

In [1]:
import pandas as pd

DATA_PATH = "../data/winequality-red.csv"
SAMPLE_PATH = "../data/wine_sample_200.csv"
SQL_PATH = "../data/wine_toscana.sql"
SAMPLE_SIZE = 200
RANDOM_STATE = 42


def load_data():
    df = pd.read_csv(DATA_PATH, sep=";")
    df.columns = [c.strip() for c in df.columns]
    return df


def quality_tier(q):
    if q <= 4:
        return "Low"
    if q <= 6:
        return "Medium"
    return "High"


## Stratified sample

200 rows, stratified by quality tier so Low/Medium/High are all represented
in the ToscanaJ subset even though High and Low are rare in the raw data.

In [2]:
def stratified_sample(df, n=SAMPLE_SIZE):
    df = df.copy()
    df["quality_tier"] = df["quality"].apply(quality_tier)
    frac = n / len(df)
    sample = (
        df.groupby("quality_tier", group_keys=False)
        .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=RANDOM_STATE).reset_index(drop=True)
    if "quality_tier" not in sample.columns:
        sample["quality_tier"] = sample["quality"].apply(quality_tier)
    sample.insert(0, "wine_id", range(1, len(sample) + 1))
    return sample


df = load_data()
sample = stratified_sample(df)
sample.to_csv(SAMPLE_PATH, index=False)
print(f"Wrote {len(sample)}-row stratified sample to {SAMPLE_PATH}")
sample["quality_tier"].value_counts()


Wrote 200-row stratified sample to ../data/wine_sample_200.csv


quality_tier
Medium    165
High       27
Low         8
Name: count, dtype: int64

## Scale definitions

Each function maps a raw value to the set of derived (ordinal / dichotomic /
interordinal) attributes it satisfies, per the attribute inventory in
`report/ex2_toscana.md` section 2. Ordinal scales use "at least bound" semantics so they
are downward-closed (a value above the highest bound also satisfies every
lower bound) — this is what makes them valid ordinal scales rather than
nominal ones.

In [3]:
def scale_fixed_acidity(v):
    out = []
    if v >= 6:
        out.append("fixed_acidity>=low(6)")
    if v >= 8:
        out.append("fixed_acidity>=medium(8)")
    if v >= 10:
        out.append("fixed_acidity>=high(10)")
    return out


def scale_volatile_acidity(v):
    out = []
    if v >= 0.3:
        out.append("volatile_acidity>=low(0.3)")
    if v >= 0.5:
        out.append("volatile_acidity>=medium(0.5)")
    if v >= 0.7:
        out.append("volatile_acidity>=high(0.7)")
    return out


def scale_citric_acid(v):
    return ["citric_acid_present"] if v >= 0.25 else ["citric_acid_absent"]


def scale_residual_sugar(v):
    # dry < 4, off-dry 4-12, sweet > 12 (g/L) — wine industry convention
    if v < 4:
        return ["residual_sugar=dry"]
    if v <= 12:
        return ["residual_sugar=off_dry"]
    return ["residual_sugar=sweet"]


def scale_chlorides(v):
    return ["chlorides=high_salt"] if v >= 0.08 else ["chlorides=low_salt"]


def scale_free_so2(v):
    out = []
    if v >= 6:
        out.append("free_so2>=low(6)")
    if v >= 15:
        out.append("free_so2>=medium(15)")
    if v >= 30:
        out.append("free_so2>=high(30)")
    return out


def scale_total_so2(v):
    out = []
    if v >= 20:
        out.append("total_so2>=low(20)")
    if v >= 60:
        out.append("total_so2>=medium(60)")
    if v >= 100:
        out.append("total_so2>=high(100)")
    return out


def scale_density(v):
    return ["density=light"] if v < 0.997 else ["density=heavy"]


def scale_pH(v):
    # interordinal: acidic side and basic side, both increasing-include-bounds
    out = []
    if v <= 3.1:
        out.append("pH<=acidic(3.1)")
    if 3.1 < v < 3.4:
        out.append("pH=neutral_range")
    if v >= 3.4:
        out.append("pH>=basic(3.4)")
    return out


def scale_sulphates(v):
    out = []
    if v >= 0.4:
        out.append("sulphates>=low(0.4)")
    if v >= 0.6:
        out.append("sulphates>=medium(0.6)")
    if v >= 0.8:
        out.append("sulphates>=high(0.8)")
    return out


def scale_alcohol(v):
    out = []
    if v >= 9:
        out.append("alcohol>=9")
    if v >= 11:
        out.append("alcohol>=11")
    if v >= 13:
        out.append("alcohol>=13")
    return out


def scale_quality(v):
    out = []
    if v >= 5:
        out.append("quality>=acceptable(5)")
    if v >= 7:
        out.append("quality>=good(7)")
    if v >= 8:
        out.append("quality>=excellent(8)")
    return out


SCALES = {
    "fixed acidity": scale_fixed_acidity,
    "volatile acidity": scale_volatile_acidity,
    "citric acid": scale_citric_acid,
    "residual sugar": scale_residual_sugar,
    "chlorides": scale_chlorides,
    "free sulfur dioxide": scale_free_so2,
    "total sulfur dioxide": scale_total_so2,
    "density": scale_density,
    "pH": scale_pH,
    "sulphates": scale_sulphates,
    "alcohol": scale_alcohol,
    "quality": scale_quality,
}


## Build the derived context

In [4]:
def build_derived_context(sample):
    rows = {}
    for _, row in sample.iterrows():
        attrs = []
        for col, scale_fn in SCALES.items():
            attrs.extend(scale_fn(row[col]))
        rows[int(row["wine_id"])] = attrs
    return rows


derived = build_derived_context(sample)
list(derived.items())[:3]


[(1,
  ['fixed_acidity>=low(6)',
   'citric_acid_present',
   'residual_sugar=dry',
   'chlorides=low_salt',
   'free_so2>=low(6)',
   'density=light',
   'pH=neutral_range',
   'sulphates>=low(0.4)',
   'sulphates>=medium(0.6)',
   'sulphates>=high(0.8)',
   'alcohol>=9',
   'alcohol>=11',
   'quality>=acceptable(5)',
   'quality>=good(7)']),
 (2,
  ['fixed_acidity>=low(6)',
   'volatile_acidity>=low(0.3)',
   'citric_acid_present',
   'residual_sugar=dry',
   'chlorides=low_salt',
   'free_so2>=low(6)',
   'free_so2>=medium(15)',
   'total_so2>=low(20)',
   'density=light',
   'pH=neutral_range',
   'sulphates>=low(0.4)',
   'sulphates>=medium(0.6)',
   'alcohol>=9',
   'alcohol>=11',
   'quality>=acceptable(5)',
   'quality>=good(7)',
   'quality>=excellent(8)']),
 (3,
  ['fixed_acidity>=low(6)',
   'fixed_acidity>=medium(8)',
   'volatile_acidity>=low(0.3)',
   'citric_acid_present',
   'residual_sugar=dry',
   'chlorides=high_salt',
   'free_so2>=low(6)',
   'total_so2>=low(20)',


## Verify scale closure

For every ordinal chain, if a wine satisfies the highest bound it must also
satisfy every lower bound. A violation here would mean a bug in the scale
function, not a property of the data.

In [5]:
def verify_scale_closure(derived):
    chains = [
        ["alcohol>=9", "alcohol>=11", "alcohol>=13"],
        ["fixed_acidity>=low(6)", "fixed_acidity>=medium(8)", "fixed_acidity>=high(10)"],
        ["volatile_acidity>=low(0.3)", "volatile_acidity>=medium(0.5)", "volatile_acidity>=high(0.7)"],
        ["free_so2>=low(6)", "free_so2>=medium(15)", "free_so2>=high(30)"],
        ["total_so2>=low(20)", "total_so2>=medium(60)", "total_so2>=high(100)"],
        ["sulphates>=low(0.4)", "sulphates>=medium(0.6)", "sulphates>=high(0.8)"],
        ["quality>=acceptable(5)", "quality>=good(7)", "quality>=excellent(8)"],
    ]
    errors = []
    for wine_id, attrs in derived.items():
        attr_set = set(attrs)
        for chain in chains:
            seen_true = False
            for a in reversed(chain):
                present = a in attr_set
                if present:
                    seen_true = True
                elif seen_true:
                    errors.append((wine_id, chain, a))
    if errors:
        raise AssertionError(f"Scale closure violated: {errors[:5]} (+{len(errors)-5} more)")
    print(f"Scale closure verified for {len(derived)} wines across {len(chains)} ordinal chains.")


verify_scale_closure(derived)


Scale closure verified for 200 wines across 7 ordinal chains.


## Enumerate all derived attributes

In [6]:
def all_attributes():
    attrs = []
    seen = set()
    for col, fn in SCALES.items():
        for probe in [0, 1, 5, 10, 20, 50, 100, 0.1, 0.3, 0.5, 0.7, 0.9, 3.0, 3.2, 3.5, 9, 11, 13]:
            for a in fn(probe):
                if a not in seen:
                    seen.add(a)
                    attrs.append(a)
    return attrs


attrs = all_attributes()
print(f"Derived context has {len(attrs)} binary attributes:")
for a in attrs:
    print(f"  - {a}")


Derived context has 33 binary attributes:
  - fixed_acidity>=low(6)
  - fixed_acidity>=medium(8)
  - fixed_acidity>=high(10)
  - volatile_acidity>=low(0.3)
  - volatile_acidity>=medium(0.5)
  - volatile_acidity>=high(0.7)
  - citric_acid_absent
  - citric_acid_present
  - residual_sugar=dry
  - residual_sugar=off_dry
  - residual_sugar=sweet
  - chlorides=low_salt
  - chlorides=high_salt
  - free_so2>=low(6)
  - free_so2>=medium(15)
  - free_so2>=high(30)
  - total_so2>=low(20)
  - total_so2>=medium(60)
  - total_so2>=high(100)
  - density=light
  - density=heavy
  - pH<=acidic(3.1)
  - pH>=basic(3.4)
  - pH=neutral_range
  - sulphates>=low(0.4)
  - sulphates>=medium(0.6)
  - sulphates>=high(0.8)
  - alcohol>=9
  - alcohol>=11
  - alcohol>=13
  - quality>=acceptable(5)
  - quality>=good(7)
  - quality>=excellent(8)


## Export to HSQLDB SQL for Elba

HSQLDB uses `VARCHAR`/`DOUBLE`, not generic `TEXT`; column names must be valid
SQL identifiers. The table carries the **raw** numeric attributes (plus
`wine_id`) — Elba reads this table and builds each scale from these columns
using the thresholds listed below.

In [7]:
def sql_identifier(name):
    return name.replace(" ", "_").replace(">", "ge").replace("<", "le").replace(
        "=", "_").replace("(", "_").replace(")", "_").replace(".", "_").replace("-", "_")


RAW_COLUMNS = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol", "quality",
]


def write_raw_sql(sample, path=SQL_PATH):
    col_map = {c: sql_identifier(c) for c in RAW_COLUMNS}
    with open(path, "w") as f:
        f.write("-- Raw many-valued context for Elba (HSQLDB)\n")
        f.write("-- Generated by notebooks/ex2_prepare_context.ipynb\n")
        f.write("-- Elba builds the conceptual scales from these raw columns.\n\n")
        f.write("DROP TABLE wine_context IF EXISTS;\n\n")
        f.write("CREATE TABLE wine_context (\n")
        f.write("  wine_id INTEGER PRIMARY KEY,\n")
        col_defs = ",\n".join(
            f"  {col_map[c]} {'INTEGER' if c == 'quality' else 'DOUBLE'}" for c in RAW_COLUMNS
        )
        f.write(col_defs + "\n);\n\n")

        cols = ", ".join(["wine_id"] + [col_map[c] for c in RAW_COLUMNS])
        for _, row in sample.iterrows():
            vals = [str(int(row["wine_id"]))] + [str(row[c]) for c in RAW_COLUMNS]
            f.write(f"INSERT INTO wine_context ({cols}) VALUES ({', '.join(vals)});\n")
    print(f"Wrote {len(sample)} rows x {len(RAW_COLUMNS)} raw attributes to {path}")


write_raw_sql(sample)


Wrote 200 rows x 12 raw attributes to ../data/wine_toscana.sql


## Scale thresholds entered in Elba

For each raw column, a scale was created in Elba using these thresholds
(mirrors the `SCALES` functions above):

| Column | Type (Elba dropdown) | Thresholds | Title |
|---|---|---|---|
| fixed_acidity | increasing, include bounds | 6, 8, 10 | Fixed Acidity |
| volatile_acidity | increasing, include bounds | 0.3, 0.5, 0.7 | Volatile Acidity |
| citric_acid | increasing, include bounds | 0.25 | Citric Acid Present |
| residual_sugar | increasing, include bounds | 4, 12 | Residual Sugar |
| chlorides | increasing, include bounds | 0.08 | Chlorides High |
| free_sulfur_dioxide | increasing, include bounds | 6, 15, 30 | Free SO2 |
| total_sulfur_dioxide | increasing, include bounds | 20, 60, 100 | Total SO2 |
| density | increasing, include bounds | 0.997 | Density Heavy |
| pH | both, increasing side includes bounds | 3.1, 3.4 | pH Range |
| sulphates | increasing, include bounds | 0.4, 0.6, 0.8 | Sulphates |
| alcohol | increasing, include bounds | 9, 11, 13 | Alcohol |
| quality | increasing, include bounds | 5, 7, 8 | Quality Tier |